# Dashboard Milho CEPEA - Versao Colab

Este notebook adapta o `dashboard.py` (Streamlit) para execucao no Google Colab.

## O que foi mantido
- Leitura do `cotacao_milho.xlsx` (abas `Diario` e `Media_Anual`)
- Modelos de regressao linear (BRL e USD)
- Previsoes para multiplos horizontes (dias uteis)
- Graficos interativos com Plotly
- Tabelas de previsoes, metricas, historico diario e medias anuais

## Diferenca para Streamlit
No lugar de controles da sidebar, este notebook usa variaveis de configuracao em celulas Python.

In [ ]:
# Dependencias para Colab
!pip -q install pandas openpyxl scikit-learn plotly

In [ ]:
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

pd.set_option('display.max_columns', 50)

XLSX = '/content/cotacao_milho.xlsx'
print('Bibliotecas carregadas.')

## 1) Carregar arquivo Excel
Se o arquivo nao estiver em `/content`, o notebook solicita upload.

In [ ]:
if not os.path.exists(XLSX):
    from google.colab import files
    print('Arquivo nao encontrado. Faca upload do cotacao_milho.xlsx...')
    uploaded = files.upload()
    if len(uploaded) == 0:
        raise FileNotFoundError('Nenhum arquivo foi enviado.')
    nome_arquivo = next(iter(uploaded.keys()))
    XLSX = f'/content/{nome_arquivo}'

print(f'Arquivo em uso: {XLSX}')

df_diario = pd.read_excel(XLSX, sheet_name='Diario', engine='openpyxl')
df_anual = pd.read_excel(XLSX, sheet_name='Media_Anual', engine='openpyxl')

df_diario['data'] = pd.to_datetime(df_diario['data'], errors='coerce')
df_anual['data'] = pd.to_datetime(df_anual['ano'].astype(str) + '-07-01', errors='coerce')

display(df_diario.head())
display(df_anual.head())

## 2) Configuracoes (equivalente a sidebar)
Edite os parametros e rode esta celula novamente quando quiser mudar filtros.

In [ ]:
# Opcoes: 'Ambas', 'R$ (BRL)', 'US$ (USD)'
moeda = 'Ambas'

# Exemplo: [3, 6] ou [1,2,3,4,5,6,7,10]
horizonte = [3, 6]

exibir_anuais = True
exibir_diarios = True
exibir_tendencia = True
exibir_previsoes = True

# Filtros de tabela
hist_data_inicio = df_diario['data'].min().date()
hist_data_fim = df_diario['data'].max().date()
colunas_hist = ['data', 'valor_rs', 'var_dia_pct', 'var_mes_pct', 'valor_usd']

print('Configuracoes aplicadas.')

In [ ]:
def construir_serie(df_d, df_a, col_anual, col_diario):
    return pd.concat([
        df_a[['data', col_anual]].rename(columns={col_anual: 'valor'}),
        df_d[['data', col_diario]].rename(columns={col_diario: 'valor'}),
    ], ignore_index=True).dropna().sort_values('data').reset_index(drop=True)


def treinar(serie):
    data_ref = serie['data'].min()
    s = serie.copy()
    s['dias'] = (s['data'] - data_ref).dt.days
    X = s['dias'].values.reshape(-1, 1)
    y = s['valor'].values
    modelo = LinearRegression().fit(X, y)
    return modelo, data_ref, s


def proximos_uteis(base, n):
    datas = []
    d = base
    while len(datas) < n:
        d += pd.Timedelta(days=1)
        if d.weekday() < 5:
            datas.append(d)
    return datas


def calc_metricas(modelo, serie):
    X = serie['dias'].values.reshape(-1, 1)
    y = serie['valor'].values
    yp = modelo.predict(X)
    return {
        'R2': round(r2_score(y, yp), 4),
        'MAE': round(mean_absolute_error(y, yp), 4),
        'Coef angular': round(float(modelo.coef_[0]), 6),
        'Intercepto': round(float(modelo.intercept_), 4),
    }


serie_rs = construir_serie(df_diario, df_anual, 'valor_rs_medio', 'valor_rs')
serie_usd = construir_serie(df_diario, df_anual, 'valor_usd_medio', 'valor_usd')

modelo_rs, ref_rs, serie_rs = treinar(serie_rs)
modelo_usd, ref_usd, serie_usd = treinar(serie_usd)

ultima = df_diario['data'].max()
previsoes = {}
for h in sorted(set(horizonte)):
    data_alvo = proximos_uteis(ultima, h)[-1]
    previsoes[h] = {
        'data': data_alvo,
        'prev_rs': modelo_rs.predict([[(data_alvo - ref_rs).days]])[0],
        'prev_usd': modelo_usd.predict([[(data_alvo - ref_usd).days]])[0],
    }

m_rs = calc_metricas(modelo_rs, serie_rs)
m_usd = calc_metricas(modelo_usd, serie_usd)

print('Modelos e previsoes calculados.')

In [ ]:
# KPIs (equivalente ao topo do dashboard)
ultimo_rs = df_diario['valor_rs'].iloc[-1]
ultimo_usd = df_diario['valor_usd'].iloc[-1]
delta_rs = df_diario['var_dia_pct'].iloc[-1]
delta_usd = ((df_diario['valor_usd'].iloc[-1] - df_diario['valor_usd'].iloc[-2]) / df_diario['valor_usd'].iloc[-2]) * 100

print('Ultima cotacao - R$:   R$ {:.2f} ({:+.2f}% dia)'.format(ultimo_rs, delta_rs))
print('Ultima cotacao - US$:  US$ {:.2f} ({:+.2f}% dia)'.format(ultimo_usd, delta_usd))
print('R2 modelo R$:  {:.4f}'.format(m_rs['R2']))
print('R2 modelo US$: {:.4f}'.format(m_usd['R2']))

In [ ]:
def fazer_grafico(serie, modelo, data_ref, prev, simbolo, cor_diario, titulo):
    corte = df_diario['data'].min()
    anuais = serie[serie['data'] < corte]
    diarios = serie[serie['data'] >= corte]

    fig = go.Figure()

    if exibir_anuais:
        fig.add_trace(go.Scatter(
            x=anuais['data'], y=anuais['valor'],
            mode='markers',
            marker=dict(color='#9E9E9E', size=12, symbol='diamond'),
            name='Medias anuais',
            hovertemplate='%{x|%d/%m/%Y}<br>' + simbolo + ' %{y:.2f}<extra>Media anual</extra>',
        ))

    if exibir_diarios:
        fig.add_trace(go.Scatter(
            x=diarios['data'], y=diarios['valor'],
            mode='lines+markers',
            line=dict(color=cor_diario, width=2),
            marker=dict(size=7),
            name='Cotacao diaria',
            hovertemplate='%{x|%d/%m/%Y}<br>' + simbolo + ' %{y:.2f}<extra>Diario</extra>',
        ))

    if exibir_tendencia:
        x_num = np.linspace(serie['dias'].min(), serie['dias'].max() + 15, 300)
        y_ten = modelo.predict(x_num.reshape(-1, 1))
        d_ten = [data_ref + pd.Timedelta(days=int(d)) for d in x_num]
        fig.add_trace(go.Scatter(
            x=d_ten, y=y_ten,
            mode='lines',
            line=dict(color='#E53935', width=2, dash='dot'),
            name='Tendencia',
            hoverinfo='skip',
        ))

    if exibir_previsoes:
        cores_prev = ['#F57C00', '#6A1B9A', '#0288D1', '#2E7D32', '#AD1457', '#00695C', '#4527A0', '#558B2F']
        for idx, (h, p) in enumerate(prev.items()):
            val = p['prev_rs'] if simbolo == 'R$' else p['prev_usd']
            cor = cores_prev[idx % len(cores_prev)]
            fig.add_trace(go.Scatter(
                x=[p['data']], y=[val],
                mode='markers+text',
                marker=dict(color=cor, size=14, symbol='diamond'),
                text=[f'+{h}d: {simbolo}{val:.2f}'],
                textposition='top right',
                name=f'Prev. +{h} dias uteis',
                hovertemplate=(
                    f'%{{x|%d/%m/%Y}}<br>{simbolo} %{{y:.2f}}'
                    f'<extra>Previsao +{h} dias uteis</extra>'
                ),
            ))

    fig.add_vline(
        x=ultima.timestamp() * 1000,
        line_dash='dash', line_color='#757575', line_width=1,
        annotation_text='Hoje', annotation_position='top',
    )

    fig.update_layout(
        title=dict(text=titulo, font=dict(size=15)),
        xaxis_title='Data',
        yaxis_title=f'Preco ({simbolo} / saca 60 kg)',
        hovermode='x unified',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        height=450,
        template='plotly_white',
    )
    return fig

mostrar_rs = moeda in ('Ambas', 'R$ (BRL)')
mostrar_usd = moeda in ('Ambas', 'US$ (USD)')

if mostrar_rs:
    fig_rs = fazer_grafico(serie_rs, modelo_rs, ref_rs, previsoes, 'R$', '#1565C0', 'Preco em R$ (BRL)')
    fig_rs.show()

if mostrar_usd:
    fig_usd = fazer_grafico(serie_usd, modelo_usd, ref_usd, previsoes, 'US$', '#00695C', 'Preco em US$ (USD)')
    fig_usd.show()

## 3) Tabelas do dashboard

In [ ]:
# Tabela de previsoes
df_prev = pd.DataFrame([
    {
        'Horizonte (dias uteis)': h,
        'Data alvo': p['data'].strftime('%d/%m/%Y'),
        'Previsao R$ (BRL)': round(p['prev_rs'], 2),
        'Previsao US$ (USD)': round(p['prev_usd'], 2),
    }
    for h, p in previsoes.items()
])

display(df_prev)

# Tabela de metricas
df_met = pd.DataFrame([
    {'Modelo': 'BRL (R$)', **m_rs},
    {'Modelo': 'USD (US$)', **m_usd},
])

display(df_met)

# Historico diario filtrado
df_hist = df_diario[
    (df_diario['data'].dt.date >= hist_data_inicio) &
    (df_diario['data'].dt.date <= hist_data_fim)
][colunas_hist].sort_values('data', ascending=False)

display(df_hist.head(30))

# Medias anuais
df_anual_filt = df_anual[['ano', 'valor_rs_medio', 'valor_usd_medio']].sort_values('ano')
display(df_anual_filt)

In [ ]:
# Opcional: exportar tabelas para consulta
saida_prev = '/content/previsoes_dashboard.csv'
saida_met = '/content/metricas_dashboard.csv'

df_prev.to_csv(saida_prev, index=False)
df_met.to_csv(saida_met, index=False)

print(f'Arquivos gerados: {saida_prev} e {saida_met}')

from google.colab import files
files.download(saida_prev)
files.download(saida_met)